In [ ]:
from pathlib import Path
import sys
import matplotlib as mpl
CAMERA_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'figure_generation_notebooks' else Path.cwd().resolve()
EXPERIMENT_ROOT = Path('/mnt/e/watermark_rev2')
FIGURETYPE1_DIR = CAMERA_ROOT / 'figuretype1'
FIGURETYPE1_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(EXPERIMENT_ROOT))
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from scipy.optimize import curve_fit
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42


In [ ]:
UNIFORM_POINTS = True
PARETO_SIZE = 15
PARETO_ALPHA = 0.7
LABEL_FONTSIZE = 8
TICK_FONTSIZE = 8
LEGEND_FONTSIZE = 8
SHOW_TICKS = True
CONFIG = {'xlims': (-1, 3.5), 'ylims': (0.85, 1.02), 'min_tpr': 0.8, 'max_curvature': 15, 'min_curvature': 0.1, 'curve_extend_left': 0.05, 'curve_extend_right': 0.05, 'min_points': 2}
COLORS = {'grid': '#1f77b4', 'dp': '#2ca02c', 'klt': '#d62728', 'opt': '#9467bd', 'dip': '#ff7f0e'}
LABELS = {'klt': 'Ours', 'dp': 'DP', 'grid': 'KGW', 'opt': 'OPT', 'dip': 'DiPmark'}
METHODS = ['grid', 'dp', 'klt', 'opt', 'dip']


In [ ]:
df_all = pd.read_csv(EXPERIMENT_ROOT / 'archive' / 'aggregated_power_kl_results_pareto.csv')


In [ ]:
def is_pareto_front(x, y, eps_x, eps_y):
    n = len(x)
    keep = np.ones(n, dtype=bool)
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            if x[j] >= x[i] + eps_x and y[j] >= y[i] + eps_y:
                keep[i] = False
                break
    return keep

def recalc_pareto(df, eps_x=0.02, eps_y=0.02):
    df = df.copy()
    df['is_pareto'] = False
    groups = df.groupby(['dataset', 'model', 'alpha', 'n', 'method'])
    for name, g in groups:
        idx = g.index
        x = g['neg_log_kl'].values
        y = g['power_empirical'].values
        pareto_mask = is_pareto_front(x, y, eps_x, eps_y)
        df.loc[idx, 'is_pareto'] = pareto_mask
    return df
df_all = recalc_pareto(df_all, eps_x=0.02, eps_y=0.02)


In [ ]:
def inverse_sigmoid(x, a, b):
    return 1.0 / (1.0 + np.exp(a * (x - b)))

def fit_inverse_sigmoid(x, y, cfg):
    valid = np.isfinite(x) & np.isfinite(y)
    x, y = (x[valid], y[valid])
    if len(x) < 2:
        return {'a': 1.0, 'b': np.median(x) if len(x) > 0 else 0, 'r2': 0, 'n_points': len(x), 'x_min': x.min() if len(x) > 0 else -1, 'x_max': x.max() if len(x) > 0 else 1}
    x_min, x_max = (x.min(), x.max())
    eps = 0.0001
    y_clip = np.clip(y, eps, 1 - eps)
    logit = np.log(y_clip / (1 - y_clip))
    m, c = np.polyfit(x, logit, 1)
    a0 = -m
    b0 = c / a0 if a0 != 0 else np.median(x)
    a0 = np.clip(abs(a0), 0.01, 20)
    try:
        popt, _ = curve_fit(inverse_sigmoid, x, y_clip, p0=[a0, b0], bounds=([0.001, x_min - 10], [50, x_max + 10]), maxfev=20000)
        a, b = popt
    except:
        a, b = (a0, b0)
    y_pred = inverse_sigmoid(x, a, b)
    ss_res = np.sum((y - y_pred) ** 2)
    ss_tot = np.sum((y - np.mean(y)) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0
    return {'a': float(a), 'b': float(b), 'r2': float(r2), 'n_points': len(x), 'x_min': x_min, 'x_max': x_max}

def fit_all_methods(df_subset, cfg):
    df = df_subset[df_subset['power_empirical'] >= cfg['min_tpr']].copy()
    results = {}
    for method in METHODS:
        df_m = df[df['method'] == method]
        if len(df_m) == 0:
            continue
        df_pareto = df_m[df_m['is_pareto'] == True]
        if len(df_pareto) < cfg['min_points']:
            continue
        x = df_pareto['neg_log_kl'].values
        y = df_pareto['power_empirical'].values
        result = fit_inverse_sigmoid(x, y, cfg)
        if result:
            results[method] = result
    return results


In [ ]:
def style_axes_clean(ax):
    ax.set_facecolor('white')
    ax.grid(False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    for side in ['left', 'bottom']:
        ax.spines[side].set_color('black')
        ax.spines[side].set_linewidth(0.8)
    if SHOW_TICKS:
        ax.tick_params(axis='both', colors='black', direction='out', length=3, width=0.8, labelsize=TICK_FONTSIZE)
    else:
        ax.tick_params(axis='both', which='both', length=0, labelbottom=False, labelleft=False)

def plot_panel(ax, subset, cfg, non_pareto_alpha=0.2, curve_alpha=0.6, show_legend=False, show_ylabel=True, min_tpr=0.8):
    if subset is None or len(subset) == 0:
        ax.set_visible(False)
        return
    subset = subset[subset['power_empirical'] > min_tpr].copy()
    if len(subset) == 0:
        ax.set_visible(False)
        return
    fit_results = fit_all_methods(subset, cfg)
    y = subset['power_empirical'].values
    x_log = subset['neg_log_kl'].values
    methods_col = subset['method'].values
    is_pareto = subset['is_pareto'].values
    if UNIFORM_POINTS:
        np_size = PARETO_SIZE
        np_alpha = PARETO_ALPHA
    else:
        np_size = 20
        np_alpha = non_pareto_alpha
    style_axes_clean(ax)
    for m in METHODS:
        m_mask = methods_col == m
        if not np.any(m_mask):
            continue
        pareto_mask = m_mask & is_pareto
        non_pareto_mask = m_mask & ~is_pareto
        if np.any(non_pareto_mask):
            sns.scatterplot(x=x_log[non_pareto_mask], y=y[non_pareto_mask], ax=ax, color=COLORS[m], s=np_size, alpha=np_alpha, edgecolor='white', linewidth=0.3, legend=False)
        if np.any(pareto_mask):
            sns.scatterplot(x=x_log[pareto_mask], y=y[pareto_mask], ax=ax, color=COLORS[m], s=PARETO_SIZE, alpha=PARETO_ALPHA, edgecolor='white', linewidth=0.3, label=LABELS[m], legend=False)
        elif np.any(non_pareto_mask):
            ax.scatter([], [], label=LABELS[m], color=COLORS[m], marker='o', s=PARETO_SIZE)
        if m in fit_results:
            res = fit_results[m]
            x_left = res['x_min'] - cfg['curve_extend_left']
            x_right = res['x_max'] + cfg['curve_extend_right']
            x_curve = np.linspace(x_left, x_right, 100)
            y_curve = inverse_sigmoid(x_curve, res['a'], res['b'])
            ax.plot(x_curve, y_curve, color=COLORS[m], linewidth=1.2, alpha=curve_alpha)
    ax.set_xlabel('$-\\log(\\mathrm{KL})$', fontsize=LABEL_FONTSIZE)
    if show_ylabel:
        ax.set_ylabel('TPR', fontsize=LABEL_FONTSIZE)
    else:
        ax.set_ylabel('')
    ax.set_xlim(cfg['xlims'])
    ax.set_ylim(cfg['ylims'])
    if show_legend:
        legend = ax.legend(loc='lower left', fontsize=LEGEND_FONTSIZE, frameon=True)
        frame = legend.get_frame()
        frame.set_facecolor('white')
        frame.set_edgecolor('#cccccc')


In [ ]:
def plot_dataset_alpha_n_vertical(df, dataset, alpha, n, cfg=None, non_pareto_alpha=0.2, curve_alpha=0.6, min_tpr=0.8, save=True, out_dir=str(FIGURETYPE1_DIR)):
    sns.set_theme(style='white')
    cfg = cfg or CONFIG
    sub = df[(df['dataset'] == dataset) & (df['alpha'] == alpha) & (df['n'] == n)].copy()
    if len(sub) == 0:
        return
    models = sorted(sub['model'].unique())
    n_rows = len(models)
    FIG_WIDTH = 6
    PANEL_HEIGHT = 2
    fig, axes = plt.subplots(n_rows, 1, figsize=(FIG_WIDTH, PANEL_HEIGHT * n_rows), squeeze=False, sharex=True, facecolor='white')
    for i, model in enumerate(models):
        subset = sub[sub['model'] == model]
        is_last = i == n_rows - 1
        show_ylabel = i == 1
        plot_panel(axes[i, 0], subset, cfg, non_pareto_alpha=non_pareto_alpha, curve_alpha=curve_alpha, show_legend=is_last, show_ylabel=show_ylabel, min_tpr=min_tpr)
        axes[i, 0].set_title(f'{model.upper()}', fontsize=LABEL_FONTSIZE)
        if not is_last:
            axes[i, 0].set_xlabel('')
    plt.tight_layout()
    if save:
        os.makedirs(out_dir, exist_ok=True)
        fname = f'{out_dir}/pareto_vertical_{dataset}_a{alpha}_n{int(n)}.pdf'
        plt.savefig(fname, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    plt.close(fig)


In [ ]:
for ds in df_all['dataset'].unique():
    plot_dataset_alpha_n_vertical(df_all, dataset=ds, alpha=0.05, n=50, non_pareto_alpha=0.7, curve_alpha=0.7)


In [ ]:
def plot_dataset_alpha_n_vertical_sns(df, dataset, alpha, n, cfg=None, non_pareto_alpha=0.2, curve_alpha=0.6, min_tpr=0.8, save=True, out_dir=str(FIGURETYPE1_DIR)):
    cfg = cfg or CONFIG
    sub = df[(df['dataset'] == dataset) & (df['alpha'] == alpha) & (df['n'] == n)].copy()
    if len(sub) == 0:
        return
    sub = sub[sub['power_empirical'] > min_tpr].copy()
    if len(sub) == 0:
        return
    models = sorted(sub['model'].unique())
    n_rows = len(models)
    FIG_WIDTH = 6
    PANEL_HEIGHT = 2
    sns.set_theme(style='whitegrid', rc={'axes.facecolor': '#fafafa', 'figure.facecolor': 'white', 'grid.color': '#e0e0e0', 'grid.linestyle': '--', 'grid.linewidth': 0.5, 'axes.edgecolor': '#333333', 'axes.linewidth': 1.2})
    COLORS_SNS = {'grid': '#4c72b0', 'dp': '#55a868', 'klt': '#c44e52', 'opt': '#8172b3', 'dip': '#ccb974'}
    fig, axes = plt.subplots(n_rows, 1, figsize=(FIG_WIDTH, PANEL_HEIGHT * n_rows), squeeze=False, sharex=True, facecolor='white')
    for i, model in enumerate(models):
        ax = axes[i, 0]
        subset = sub[sub['model'] == model].copy()
        if len(subset) == 0:
            ax.set_visible(False)
            continue
        fit_results = fit_all_methods(subset, cfg)
        is_last = i == n_rows - 1
        show_ylabel = i == 1
        for m in METHODS:
            m_mask = subset['method'] == m
            if not np.any(m_mask):
                continue
            df_m = subset[m_mask]
            pareto_mask = df_m['is_pareto'] == True
            non_pareto_mask = ~pareto_mask
            if np.any(non_pareto_mask):
                df_np = df_m[non_pareto_mask]
                ax.scatter(df_np['neg_log_kl'], df_np['power_empirical'], c=COLORS_SNS[m], s=PARETO_SIZE, alpha=non_pareto_alpha, edgecolor=COLORS_SNS[m], linewidth=0.8, marker='o', facecolors='none', zorder=2)
            if np.any(pareto_mask):
                df_p = df_m[pareto_mask]
                ax.scatter(df_p['neg_log_kl'], df_p['power_empirical'], c=COLORS_SNS[m], s=PARETO_SIZE * 1.2, alpha=PARETO_ALPHA, edgecolor='white', linewidth=0.8, marker='o', label=LABELS[m], zorder=3)
            elif np.any(non_pareto_mask):
                ax.scatter([], [], label=LABELS[m], color=COLORS_SNS[m], marker='o', s=PARETO_SIZE)
            if m in fit_results:
                res = fit_results[m]
                x_left = res['x_min'] - cfg['curve_extend_left']
                x_right = res['x_max'] + cfg['curve_extend_right']
                x_curve = np.linspace(x_left, x_right, 200)
                y_curve = inverse_sigmoid(x_curve, res['a'], res['b'])
                ax.plot(x_curve, y_curve, color=COLORS_SNS[m], linewidth=2.5, alpha=curve_alpha, zorder=4)
                ax.plot(x_curve, y_curve, color=COLORS_SNS[m], linewidth=4, alpha=0.15, zorder=3)
        ax.set_xlim(cfg['xlims'])
        ax.set_ylim(cfg['ylims'])
        ax.set_title(f'{model.upper()}', fontsize=LABEL_FONTSIZE, fontweight='bold', color='#333333', pad=8)
        if is_last:
            ax.set_xlabel('$-\\log(\\mathrm{KL})$', fontsize=LABEL_FONTSIZE, color='#333333')
        else:
            ax.set_xlabel('')
        if show_ylabel:
            ax.set_ylabel('TPR', fontsize=LABEL_FONTSIZE, color='#333333')
        else:
            ax.set_ylabel('')
        ax.tick_params(axis='both', labelsize=TICK_FONTSIZE, colors='#333333')
        if is_last:
            legend = ax.legend(loc='lower left', fontsize=LEGEND_FONTSIZE, frameon=True, fancybox=True, shadow=False, borderpad=0.8, labelspacing=0.5, handletextpad=0.5, columnspacing=1.0)
            frame = legend.get_frame()
            frame.set_facecolor('white')
            frame.set_edgecolor('#cccccc')
            frame.set_alpha(0.95)
            frame.set_linewidth(0.8)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_color('#666666')
        ax.spines['bottom'].set_color('#666666')
    plt.tight_layout(h_pad=1.5)
    if save:
        os.makedirs(out_dir, exist_ok=True)
        fname = f'{out_dir}/pareto_vertical_sns_{dataset}_a{alpha}_n{int(n)}.pdf'
        plt.savefig(fname, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    plt.close(fig)
    sns.set_theme(style='white')
for ds in df_all['dataset'].unique():
    plot_dataset_alpha_n_vertical_sns(df_all, dataset=ds, alpha=0.05, n=50, non_pareto_alpha=0.4, curve_alpha=0.8)


In [ ]:
def plot_all_datasets_3x3(df, alpha, n, cfg=None, non_pareto_alpha=0.2, curve_alpha=0.6, min_tpr=0.8, save=True, out_dir=str(FIGURETYPE1_DIR), legend_fontsize=6):
    sns.set_theme(style='white')
    cfg = cfg or CONFIG
    sub = df[(df['alpha'] == alpha) & (df['n'] == n)].copy()
    if len(sub) == 0:
        return
    datasets = sorted(sub['dataset'].unique())
    models = sorted(sub['model'].unique())
    n_rows = len(models)
    n_cols = len(datasets)
    PANEL_WIDTH = 2.5
    PANEL_HEIGHT = 1.3
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(PANEL_WIDTH * n_cols, PANEL_HEIGHT * n_rows), squeeze=False, sharex=True, sharey=True, facecolor='white')
    for i, model in enumerate(models):
        for j, dataset in enumerate(datasets):
            ax = axes[i, j]
            subset = sub[(sub['model'] == model) & (sub['dataset'] == dataset)]
            is_last_row = i == n_rows - 1
            is_first_col = j == 0
            is_bottom_left = i == n_rows - 1 and j == 0
            is_bottom_middle = i == n_rows - 1 and j == n_cols // 2
            show_ylabel = is_first_col and i == n_rows // 2
            plot_panel(ax, subset, cfg, non_pareto_alpha=non_pareto_alpha, curve_alpha=curve_alpha, show_legend=is_bottom_left, show_ylabel=show_ylabel, min_tpr=min_tpr)
            if is_bottom_left:
                legend = ax.get_legend()
                if legend is not None:
                    handles, labels = ax.get_legend_handles_labels()
                    ax.legend_.remove()
                    legend = ax.legend(handles, labels, loc='lower left', fontsize=legend_fontsize, frameon=True, borderpad=0.3, labelspacing=0.2, handletextpad=0.3, handlelength=1.0, markerscale=0.6)
                    frame = legend.get_frame()
                    frame.set_facecolor('white')
                    frame.set_edgecolor('#cccccc')
            if i == 0:
                ax.set_title(f'{dataset.upper()}', fontsize=LABEL_FONTSIZE, pad=2)
            if j == n_cols - 1:
                ax.annotate(f'{model.upper()}', xy=(1.02, 0.5), xycoords='axes fraction', fontsize=LABEL_FONTSIZE, ha='left', va='center', rotation=270)
            if not is_bottom_middle:
                ax.set_xlabel('')
            if not is_first_col:
                ax.set_ylabel('')
    plt.tight_layout(pad=0.3, h_pad=0.4, w_pad=0.4)
    plt.subplots_adjust(right=0.91)
    if save:
        os.makedirs(out_dir, exist_ok=True)
        fname = f'{out_dir}/pareto_3x3_a{alpha}_n{int(n)}.pdf'
        plt.savefig(fname, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    plt.close(fig)
plot_all_datasets_3x3(df_all, alpha=0.05, n=50, non_pareto_alpha=0.7, curve_alpha=0.7, legend_fontsize=6)
